<a href="https://colab.research.google.com/github/safdarmarwat/colab/blob/main/eng_urdu_trans01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install required libraries
!pip install gradio transformers torch sentencepiece sacremoses

# Cell 2: Import libraries and load model
import gradio as gr
from transformers import MarianMTModel, MarianTokenizer
import torch

# Load the English to Urdu translation model
model_name = "Helsinki-NLP/opus-mt-en-ur"
print("Loading English to Urdu translation model...")
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")

# Cell 3: Define translation function
def translate_to_urdu(english_text):
    """
    Translate English text to Urdu
    """
    if not english_text.strip():
        return "Please enter some text to translate."

    # Tokenize input
    inputs = tokenizer(english_text, return_tensors="pt", truncation=True, padding=True)

    # Move to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate translation
    with torch.no_grad():
        translated = model.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)

    # Decode the translation
    urdu_text = tokenizer.decode(translated[0], skip_special_tokens=True)

    return urdu_text

# Cell 4: Create Gradio interface
# Example sentences for demonstration
examples = [
    "Hello, how are you?",
    "I love learning new languages.",
    "What is your name?",
    "The weather is beautiful today.",
    "Can you help me please?",
    "I am going to the market.",
    "This is a wonderful day.",
    "Thank you very much.",
    "Where is the nearest hospital?",
    "I would like to order some food."
]

# Custom CSS for better UI
custom_css = """
.gradio-container {
    max-width: 800px !important;
    margin: auto !important;
}
h1 {
    text-align: center;
    color: #2c3e50;
    font-size: 2.5em;
    margin-bottom: 0.5em;
}
.description {
    text-align: center;
    margin-bottom: 2em;
    color: #34495e;
}
"""

# Create the interface
iface = gr.Interface(
    fn=translate_to_urdu,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter English text here...",
        label="English Text",
        elem_id="input-text"
    ),
    outputs=gr.Textbox(
        lines=3,
        placeholder="Urdu translation will appear here...",
        label="Urdu Translation",
        elem_id="output-text"
    ),
    title="English to Urdu Translation App",
    description="""
    <div class='description'>
    This app uses Helsinki-NLP's MarianMT model to translate English text to Urdu.
    Enter any English sentence and get instant Urdu translation.
    <br><br>
    <strong>Examples of common phrases:</strong>
    </div>
    """,
    examples=examples,
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"),
    css=custom_css,
    allow_flagging="never"
)

# Cell 5: Launch the app
# For Colab, we need to use share=True to create a public link
iface.launch(share=True, debug=True)

# Cell 6: Alternative - Create a more advanced version with additional features
def advanced_translation(english_text, num_beams=4, max_length=512):
    """
    Advanced translation with adjustable parameters
    """
    if not english_text.strip():
        return "Please enter some text to translate.", ""

    # Tokenize input
    inputs = tokenizer(english_text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate translation with custom parameters
    with torch.no_grad():
        translated = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
            temperature=0.7 if num_beams == 1 else None
        )

    # Decode translation
    urdu_text = tokenizer.decode(translated[0], skip_special_tokens=True)

    # Character count
    char_count = len(urdu_text)

    return urdu_text, f"Character count: {char_count}"

# Create advanced interface
advanced_iface = gr.Interface(
    fn=advanced_translation,
    inputs=[
        gr.Textbox(lines=3, placeholder="Enter English text here...", label="English Text"),
        gr.Slider(minimum=1, maximum=8, value=4, step=1, label="Number of Beams (higher = better quality but slower)"),
        gr.Slider(minimum=128, maximum=512, value=256, step=32, label="Max Length")
    ],
    outputs=[
        gr.Textbox(lines=3, label="Urdu Translation"),
        gr.Textbox(label="Info")
    ],
    title="Advanced English to Urdu Translation",
    description="Enhanced translation with adjustable parameters for better control",
    theme=gr.themes.Soft()
)

# Launch advanced version (uncomment to use)
# advanced_iface.launch(share=True)

# Cell 7: Test the translation
test_sentences = [
    "Hello, welcome to our translation app.",
    "I am learning Urdu language.",
    "This is a beautiful day."
]

print("Testing translation model:")
print("-" * 50)
for sentence in test_sentences:
    translation = translate_to_urdu(sentence)
    print(f"English: {sentence}")
    print(f"Urdu: {translation}")
    print("-" * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 16.4 MB/s eta 0:00:00
Loading English to Urdu translation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/816k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/848k [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded on cpu


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://86e150544c481623ca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
